# Exercise 3 — Hazard classification from exceedance probabilities — Barbados

This notebook is the analysis behind the dashboard you build in
**[docs/Barbados/exercise_3.rst](../../docs/Barbados/exercise_3.rst)**. The four
probability rasters you put on the map in
[Exercise 1](../../docs/Barbados/exercise_1.rst) are combined here into a single hazard
classification, and then used to say which buildings and roads fall inside each level —
the same computation the `Barbados Hands On 3` dashboard runs behind its three tiles.

The question being answered is:

> **What are the odds of at least 30 cm of water here, and what does that classify as?**

[Exercise 2](02_storm_impact.ipynb) sampled *depth* from one storm of a parish library.
This works from **exceedance probabilities** across the 50 members of a forecast, which
is the product a forecaster actually issues.

**The point of the notebook.** Sections 1 to 7 reproduce what the dashboard shows.
**Section 8 goes past it** — the same four columns, turned into three charts no plugin
ships, because once the data is in a dataframe it will answer whatever you ask it. That
is the skill worth taking away.

**Workflow**

1. Read the four probability rasters and check they share a grid
2. Classify pixels into hazard levels
3. Sweep a gate to see what the data can actually reach
4. Classify the buildings and roads directly from the probabilities they carry
5. Recover the rule behind the file's own `hazard_flag` column
6. Build the impact table — island-wide and by parish
7. Map it
8. **Extend it**: three visualizations the dashboard does not ship
9. From notebook to dashboard: which plugin does which section

Everything is read from a public S3 bucket. Only the receptor geopackage is downloaded,
and only once — the same file Exercise 2 uses, so if you have run that notebook it is
already cached.

In [ ]:
!pip install -q rasterio pyogrio

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

%matplotlib inline

BUCKET = "https://cog-s3-test-401506828094-us-east-1-an.s3.us-east-1.amazonaws.com"
ROOT = f"{BUCKET}/Barbados_training/Barbados_Tomas_2010_flood_maps_for_IBF"
CYCLE = "20101030.000000"

# P(maximum depth >= threshold) for the four IBF depth thresholds, shallowest first.
# These are the eleven parish products mosaicked onto one island grid -- the same four
# layers exercise 1 puts on the map.
ORDER = ["10cm", "30cm", "70cm", "100cm"]
PROB = {d: f"{ROOT}/03_fim_island_mosaic/prob_depth_ge_{d}.{CYCLE}.tif" for d in ORDER}

# Buildings and roads for the whole island, each receptor counted once.
FEATURES = f"{ROOT}/04_ibf_island_deduplicated/barbados_ibf_receptors_full.{CYCLE}.gpkg"

print("ready")

## 1. The four probability rasters

Each raster holds **P(maximum flood depth ≥ threshold)** for one depth. The forecast that
produced them is the Hurricane Tomas hindcast of 30 October 2010, run with **50 rainfall
members**. Each member's rainfall total over a parish selects the closest scenario from
that parish's library of 200 synthetic storms — the library [Exercise
2](02_storm_impact.ipynb) reads — and the probability at a cell is the share of the 50
members whose matched scenario floods it that deeply.

A value of 0.30 in the 30 cm raster therefore means *"15 of the 50 members put at least
30 cm of water here"*. Two consequences follow from that construction, and both show up
below:

- probabilities move in steps of **1/50 = 0.02** and take few distinct values
- a footprint every matched scenario shares reaches probability **1.0**, so the top
  hazard level is reachable

Why the mosaic rather than the eleven parish windows? The windows are parish-plus-buffer
and overlap, and in the overlap neighbouring products disagree, because each was matched
on its own parish's rainfall. For anything island-wide, use the mosaic.

In [ ]:
layers, meta = {}, {}
for name, url in PROB.items():
    with rasterio.open(url) as ds:
        layers[name] = ds.read(1, masked=True)
        meta[name] = {"crs": ds.crs, "shape": ds.shape, "transform": ds.transform,
                      "res_deg": ds.res[0], "nodata": ds.nodata}

for name, m in meta.items():
    valid = layers[name].compressed()
    print(f"P(>= {name:>5})  {m['crs']}  {m['shape']}  {m['res_deg'] * 3600:.0f} arc-second  "
          f"{(valid > 0).sum():>7,} cells above zero  {len(np.unique(valid)):>3} distinct values  "
          f"max {valid.max():.2f}")

print(f"\nall four share one grid: {len({v.shape for v in layers.values()}) == 1}")

The rasters are EPSG:4326 at one arc-second, about 30 m. Nothing needs reprojecting, and
nothing has to be registered before a map will draw it — which is why Exercise 1 could
point a layer straight at these URLs.

Note how the counts fall away: about 80,000 cells have some chance of 10 cm, but only
10,000 have any chance of a metre. Each layer is a subset of the one before it, which is
what makes the nested classification in section 2 sensible.

## 2. The classification rule

Each hazard level is tied to **one** depth layer, and each has its own probability gate:

| level | driven by | colour | dashboard default |
|---|---|---|---|
| Low | P(≥ 10 cm) | green | 0.8 |
| Medium | P(≥ 30 cm) | yellow | 0.8 |
| High | P(≥ 70 cm) | red | 0.8 |
| Severe | P(≥ 100 cm) | purple | 0.8 |

A pixel takes the level of the **deepest** threshold whose gate it clears. That falls out
of assigning the levels in ascending order and letting later assignments win.

The gates do not have to be equal, and in other countries they are not — a deeper
threshold is rarer, so a lower gate on it says *"treat a one-in-ten chance of 70 cm as
seriously as a one-in-three chance of ankle depth"*. Barbados can afford 0.8 across the
board because every layer here reaches 1.0 somewhere. **Check what range a layer can
actually reach before you expose a control for it** — section 3 is that check.

In [ ]:
LEVELS = [(1, "green"), (2, "yellow"), (3, "red"), (4, "purple")]
LEVEL_LABEL = {0: "Normal", 1: "Low", 2: "Medium", 3: "High", 4: "Severe"}
NODATA = 255

LOW_THRESHOLD = 0.80       # gate on P(>= 10 cm)
MEDIUM_THRESHOLD = 0.80    # gate on P(>= 30 cm)
HIGH_THRESHOLD = 0.80      # gate on P(>= 70 cm)
SEVERE_THRESHOLD = 0.80    # gate on P(>= 100 cm)
GATES = {1: LOW_THRESHOLD, 2: MEDIUM_THRESHOLD, 3: HIGH_THRESHOLD, 4: SEVERE_THRESHOLD}


def classify_hazard(layers, gates):
    """0 = Normal, 1..4 = hazard level, 255 = NoData."""
    first = layers[ORDER[0]]
    # A pixel missing from any layer cannot be classified at all.
    invalid = np.zeros(first.shape, dtype=bool)
    for name in ORDER:
        invalid |= np.ma.getmaskarray(layers[name])

    hazard = np.zeros(first.shape, dtype=np.uint8)
    # Ascending order: each assignment overwrites the last, so the deepest
    # threshold a pixel clears is the one that sticks.
    for name, (value, _colour) in zip(ORDER, LEVELS):
        hazard[layers[name].filled(-np.inf) >= gates[value]] = value

    hazard[invalid] = NODATA
    return hazard


hazard = classify_hazard(layers, GATES)

counts = pd.Series(hazard[hazard != NODATA]).value_counts().sort_index()
print(counts.rename(LEVEL_LABEL).rename_axis("level").to_frame("cells").to_string())

cmap = ListedColormap(["#f0f0f0", "green", "yellow", "red", "purple"])
norm = BoundaryNorm([0, 1, 2, 3, 4, 5], cmap.N)

fig, ax = plt.subplots(figsize=(6, 8))
drawn = np.where(hazard == NODATA, np.nan, hazard)
im = ax.imshow(drawn, cmap=cmap, norm=norm, interpolation="nearest")
ax.set_title("Hazard classification, Barbados\n(gates at 0.8)")
ax.set_xticks([]); ax.set_yticks([])
cbar = fig.colorbar(im, ax=ax, ticks=[0.5, 1.5, 2.5, 3.5, 4.5], shrink=0.6)
cbar.ax.set_yticklabels([LEVEL_LABEL[v] for v in range(5)])
plt.show()

## 3. What a gate can and cannot do

Before exposing a threshold as a control someone will drag, look at what the data can
reach. Sweep one common gate across every level and watch the cell counts: with
probabilities in steps of 0.02 the counts move in plateaus, and a level empties the moment
its gate passes that layer's maximum.

Here every layer reaches 1.0, so every level survives all the way to the last step. That
is not universal — the equivalent Guatemala layer tops out at 0.2, and its Severe class
disappears entirely above that gate, so a dashboard offering a 0.8 control there would let
someone silently empty a hazard level and never know why.

In [ ]:
sweep = np.round(np.arange(0.02, 1.001, 0.02), 2)
records = []
for gate in sweep:
    trial = classify_hazard(layers, {1: gate, 2: gate, 3: gate, 4: gate})
    valid = trial[trial != NODATA]
    records.append({"gate": gate, **{LEVEL_LABEL[v]: int((valid == v).sum()) for v, _c in LEVELS}})
sweep_df = pd.DataFrame(records).set_index("gate")

fig, ax = plt.subplots(figsize=(7, 3.5))
for value, colour in LEVELS:
    ax.plot(sweep_df.index, sweep_df[LEVEL_LABEL[value]], color=colour, lw=1.5,
            label=LEVEL_LABEL[value])
ax.axvline(LOW_THRESHOLD, color="black", ls="--", lw=1, label="dashboard default")
ax.set_yscale("log")
ax.set_xlabel("gate applied to every level"); ax.set_ylabel("cells in level")
ax.set_title("Cells per hazard level as one common gate rises")
ax.legend(fontsize=8); plt.show()

sweep_df.loc[[0.1, 0.2, 0.4, 0.5, 0.8, 1.0]]

## 4. Classifying the buildings and roads

Every building and road in the receptor geopackage **already carries the four
probabilities**, sampled as the maximum over its own footprint, so the same gates classify
a feature directly with no rasterising involved.

Three things about the file shape what you can say with it:

- **It is the full island stock** — 204,727 buildings and 22,509 road segments, with
  nothing removed by severity. A count from it is exposure for Barbados, not exposure
  within a subset something else already flagged.
- **Each receptor appears once.** The run writes per-parish impact folders on
  parish-plus-buffer windows that overlap, so a receptor near a boundary can appear in up
  to five of them. This file keeps each once, from its own parish's window — which is what
  makes filtering on `ADM1_PCODE` give a clean parish count.
- **Buildings are footprints**, so a building spanning several cells is classified on the
  worst of them.

The probability columns are named after the depth (`p_ge_10cm`) rather than the hazard
level, so they are renamed on load, exactly as the plugin does.

In [ ]:
import os
import tempfile
import urllib.request
from pathlib import Path

import pyogrio

# The file names its columns after the depth; the plugins name them after the level.
RENAMES = {"p_ge_10cm": "probability_low", "p_ge_30cm": "probability_med",
           "p_ge_70cm": "probability_high", "p_ge_100cm": "probability_severe"}
PROB_FIELDS = list(RENAMES.values())
MEASURES = ["population_per_building", "building_area_m2", "road_length_m"]

BB_LAYERS = {
    "building": ("buildings_ibf",
                 ["ADM1_PCODE", "ADM1_EN", "subtype", "critical", "hazard_flag",
                  "risk_level", "population_per_building", "building_area_m2"]),
    "road": ("roads_ibf",
             ["ADM1_PCODE", "road_class", "hazard_flag", "risk_level", "road_length_m"]),
}


def cached(url):
    """Fetch once into the temp dir and return the local path.

    The transfer lands on a staging name and is moved into place only once it is
    complete, so an interrupted download cannot leave a truncated file cached under
    the real name -- GDAL reports that one as "database disk image is malformed" on
    every later run, and deleting it by hand is the only cure.
    """
    path = Path(tempfile.gettempdir()) / url.rsplit("/", 1)[-1]
    if not path.exists():
        fd, staged = tempfile.mkstemp(dir=path.parent, prefix=f"{path.name}.", suffix=".part")
        os.close(fd)
        try:
            print(f"downloading {path.name} ... (about 80 MB, once)")
            urllib.request.urlretrieve(url, staged)
            os.replace(staged, path)
        finally:
            Path(staged).unlink(missing_ok=True)
    return path


def read_features(read_geometry=False, where=None):
    """Buildings and roads stacked into one frame with a `type` column."""
    source = cached(FEATURES)
    parts = []
    for type_, (layer, columns) in BB_LAYERS.items():
        part = pyogrio.read_dataframe(source, layer=layer, columns=columns + list(RENAMES),
                                      where=where, read_geometry=read_geometry, use_arrow=True)
        part = part.rename(columns=RENAMES)
        part["type"] = type_
        parts.append(part)
    if read_geometry:
        stacked = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    else:
        stacked = pd.concat(parts, ignore_index=True)
    # A road has no population and a building no length: fill so every row sums.
    for column in MEASURES:
        stacked[column] = stacked[column].fillna(0.0) if column in stacked else 0.0
    return stacked


features = read_features()
print(f"{len(features):,} features")
print(features["type"].value_counts().to_string())
print()
features[["type", "ADM1_PCODE"] + PROB_FIELDS].head(3)

In [ ]:
def classify_features(df, gates):
    hazard = pd.Series(0, index=df.index, dtype="uint8")
    for field, (value, _colour) in zip(PROB_FIELDS, LEVELS):
        hazard[df[field] >= gates[value]] = value      # deepest gate cleared wins
    return hazard


features["hazard"] = classify_features(features, GATES)
features["level"] = features.hazard.map(LEVEL_LABEL)
pd.crosstab(features["level"], features["type"]).reindex([LEVEL_LABEL[v] for v in range(5)])

## 5. Recovering the rule behind `hazard_flag`

The geopackage carries a `hazard_flag` column written by the IBF workflow. It is a
classification someone else made, and a classification is a *claim* about how it was made
— so rather than taking it on trust, recover the rule: apply our own rule with one common
gate, sweep it, and see which value reproduces the column exactly.

This generalises well beyond this file. Whenever a dataset arrives with a category already
in it, find out what produced it before you build on it.

In [ ]:
match = {}
for gate in [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.80]:
    trial = classify_features(features, {1: gate, 2: gate, 3: gate, 4: gate})
    match[gate] = (trial == features.hazard_flag).mean()

report = pd.Series(match, name="share of features matching hazard_flag").rename_axis("common gate")
print(report.map("{:.4%}".format).to_string())

**0.50 reproduces it exactly** — every one of the 227,236 features. So `hazard_flag` is
*"the deepest threshold whose probability is at least one in two"*, and now you know that
rather than believing it.

Notice that the dashboard's default of 0.8 is a stricter reading than the one baked into
the file. Neither is wrong; they answer slightly different questions, and the gap between
them is exactly what the four number inputs on the exercise 3 dashboard exist to let
someone explore.

## 6. The impact table

This is the **summary tile** of the exercise 3 dashboard. Deepest level first; Normal is
left out because it is everything the gates did not catch, so it carries no exposure.

The first table is island-wide, against the 269,090 people the receptor file accounts for.
The second breaks the same exposure down **by parish**, each against its own population,
because a parish disaster coordinator and the national office need different denominators.
The dashboard's own table shows one parish at a time — this shows all eleven at once, and
it is the first small step past what the dashboard does.

In [ ]:
TOTAL_POPULATION = features.population_per_building.sum()      # every building on the island
print(f"population across all buildings: {TOTAL_POPULATION:,.0f}")


def impact_row(name, group, total_population=TOTAL_POPULATION):
    in_level = group[group["type"] == "building"]
    return {
        "Level": name,
        "Buildings": f"{len(in_level):,}",
        "Population": f"{group.population_per_building.sum():,.0f}",
        "Area (m²)": f"{group.building_area_m2.sum():,.0f}",
        "Roads (km)": f"{group.road_length_m.sum() / 1000:,.1f}",
        "% of population": f"{100 * group.population_per_building.sum() / total_population:.2f}%",
    }


at_risk = features[features.hazard > 0]
table = [impact_row(LEVEL_LABEL[v], at_risk[at_risk.hazard == v]) for v, _c in reversed(LEVELS)]
table.append(impact_row("TOTAL at risk", at_risk))
pd.DataFrame(table).set_index("Level")

In [ ]:
# The parish name travels on the buildings layer only, so build the lookup from there.
parish_name = (features.dropna(subset=["ADM1_EN"])
               .drop_duplicates("ADM1_PCODE").set_index("ADM1_PCODE").ADM1_EN)
parish_population = (features[features["type"] == "building"]
                     .groupby("ADM1_PCODE").population_per_building.sum())

rows = []
for pcode, group in at_risk.groupby("ADM1_PCODE"):
    row = impact_row(parish_name[pcode], group, total_population=parish_population[pcode])
    row["Worst level"] = LEVEL_LABEL[int(group.hazard.max())]
    rows.append(row)

by_parish = pd.DataFrame(rows).rename(columns={"Level": "Parish",
                                               "% of population": "% of parish population"})
by_parish.set_index("Parish").sort_values(
    "Buildings", key=lambda s: s.str.replace(",", "").astype(int), ascending=False)

## 7. Mapping it

This is the pair of **map layers** on the exercise 3 dashboard: the classified grid
underneath, the classified buildings and roads on top. Zoomed to Saint Michael, where
most of the exposure is.

Reading the geometry for one parish rather than all eleven keeps 165,000 geometries from
ever being built — the same reason the plugin filters in the driver rather than afterwards.

In [ ]:
from rasterio.transform import array_bounds

PARISH_PCODE = "BB08"       # Saint Michael; try BB01 (Christ Church) or BB10 (Saint Philip)

geo = read_features(read_geometry=True, where=f"ADM1_PCODE = '{PARISH_PCODE}'")
geo["hazard"] = classify_features(geo, GATES)
geo = geo.to_crs(meta[ORDER[0]]["crs"])       # same CRS or the two layers will not line up

grid_rows, grid_cols = meta[ORDER[0]]["shape"]
left, bottom, right, top = array_bounds(grid_rows, grid_cols, meta[ORDER[0]]["transform"])
xmin, ymin, xmax, ymax = geo.total_bounds

fig, ax = plt.subplots(figsize=(9, 8))
# Normal and NoData left transparent so the classified areas read clearly.
shaded = np.where((hazard > 0) & (hazard != NODATA), hazard, np.nan).astype("float32")
ax.imshow(shaded, cmap=cmap, norm=norm, interpolation="nearest",
          extent=(left, right, bottom, top), alpha=0.5)

for value, colour in LEVELS:
    subset = geo[geo.hazard == value]
    if not subset.empty:
        subset.plot(ax=ax, color=colour, linewidth=1.2, edgecolor=colour)

ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
ax.set_title(f"{parish_name[PARISH_PCODE]}: hazard classification with buildings and roads at risk")
ax.set_xticks([]); ax.set_yticks([])
# Buildings draw as polygon collections and roads as lines, so neither makes a legend
# entry matplotlib can reuse. Proxy patches instead.
ax.legend(handles=[Patch(facecolor=c, edgecolor="black", lw=0.4, label=LEVEL_LABEL[v])
                   for v, c in LEVELS], loc="lower left", fontsize=8)
plt.show()

## 8. Extending it — three charts the dashboard does not ship

Sections 1 to 7 reproduce the dashboard. Now the part that matters most.

You have a dataframe with one row per building and road, carrying four probabilities, a
hazard level, a parish, a population, an area, a length, a road class and a `critical`
flag. **Nothing stops you from asking it a different question.** Three examples, a few
lines each, no new data:

| chart | question it answers | who asks it |
|---|---|---|
| all eleven parishes, two ways | who do I warn, and is my ranking an artefact of parish size | a national duty officer |
| exposure against the gate | how much does my answer depend on where I set the threshold | anyone defending a number in a meeting |
| critical facilities and the road network | which of the things that must keep working are at risk | an operations lead |

In [ ]:
# --- Chart 1: every parish, count and share side by side --------------------
# The dashboard shows one parish at a time, because a plugin answers one request.
# A notebook has no such constraint.
per_parish = (at_risk.groupby("ADM1_PCODE")
              .agg(population=("population_per_building", "sum"),
                   buildings=("type", lambda s: (s == "building").sum()),
                   roads_km=("road_length_m", lambda s: s.sum() / 1000)))
per_parish["share"] = 100 * per_parish.population / parish_population.reindex(per_parish.index)
per_parish.index = per_parish.index.map(parish_name)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
count_order = per_parish.sort_values("population")
axes[0].barh(count_order.index, count_order.population, color="#4c78a8")
axes[0].set_xlabel("people in buildings at hazard level 1-4")
axes[0].set_title("How many people — absolute")

share_order = per_parish.sort_values("share")
bars = axes[1].barh(share_order.index, share_order.share, color="#e45756")
axes[1].set_xlabel("% of the parish's own population")
axes[1].set_title("How many people — as a share of the parish")
for bar, value in zip(bars, share_order.share):
    axes[1].text(value, bar.get_y() + bar.get_height() / 2, f" {value:.1f}%", va="center", fontsize=8)

fig.suptitle(f"Hurricane Tomas hindcast, gates at {LOW_THRESHOLD}: exposure by parish")
fig.tight_layout(); plt.show()

**The two panels disagree, and the disagreement is the finding.** Saint Michael leads the
absolute count because that is where the people are. Ranked by share of its own
population it drops to third, behind Christ Church and behind **Saint Peter — fifth by
count, first by share**, with one in ten of its people in a building at some hazard
level. A response planned off the left panel alone would send everything to Saint Michael
and never notice Saint Peter.

Neither panel is more correct. Which denominator belongs on the screen is a decision the
person looking at it has to make, and no plugin can make it for them.

In [ ]:
# --- Chart 2: how much does the answer depend on the gate? ------------------
# Section 3 asked this of pixels. Asking it of *people* is the version a
# decision-maker can act on, and it is the honest companion to any single number
# quoted out of section 6.
gate_sweep = np.round(np.arange(0.1, 1.001, 0.05), 2)
curve = []
for gate in gate_sweep:
    hz = classify_features(features, {1: gate, 2: gate, 3: gate, 4: gate})
    hit = features[hz > 0]
    curve.append({"gate": gate,
                  "population": hit.population_per_building.sum(),
                  "buildings": int((hit["type"] == "building").sum()),
                  "roads_km": hit.road_length_m.sum() / 1000})
curve = pd.DataFrame(curve).set_index("gate")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(curve.index, curve.population, marker="o", ms=3, lw=1.5, color="#4c78a8")
ax.axvline(0.50, color="#888888", ls=":", lw=1.2)
ax.axvline(LOW_THRESHOLD, color="black", ls="--", lw=1.2)
ax.text(0.50, curve.population.max(), " hazard_flag (0.50)", fontsize=8, va="top", color="#555555")
ax.text(LOW_THRESHOLD, curve.population.max(), " dashboard (0.80)", fontsize=8, va="top")
ax.set_xlabel("common probability gate"); ax.set_ylabel("people at some hazard level")
ax.set_title("How many people are 'at risk' depends on where you put the gate")
ax.grid(alpha=0.3)
plt.show()

span = curve.population.loc[[0.10, 0.50, LOW_THRESHOLD, 1.00]]
print(span.map("{:,.0f}".format).rename_axis("gate").to_frame("people at risk").to_string())
print(f"\nThe loosest gate puts {span.loc[0.10] / span.loc[1.00]:.1f}x as many people at "
      f"risk as the strictest. Quote a number, quote the gate with it.")

In [ ]:
# --- Chart 3: the things that have to keep working --------------------------
# `critical` flags hospitals, schools, fire and police stations. There are 528 on
# the island, each one matters individually, and that is exactly the case a
# percentage hides and a count makes visible.
critical = features[(features["type"] == "building") & (features.critical.astype(bool))]
critical_by_level = critical.level.value_counts().reindex(
    [LEVEL_LABEL[v] for v in range(5)], fill_value=0)

roads = features[features["type"] == "road"]
road_km = (roads[roads.hazard > 0]
           .assign(km=lambda d: d.road_length_m / 1000)
           .pivot_table(index="road_class", columns="level", values="km", aggfunc="sum", fill_value=0)
           .reindex(columns=[LEVEL_LABEL[v] for v, _c in LEVELS], fill_value=0))
road_km = road_km.loc[road_km.sum(axis=1).sort_values().index]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

colours = ["#d9d9d9"] + [c for _v, c in LEVELS]
axes[0].bar(critical_by_level.index, critical_by_level.values, color=colours,
            edgecolor="black", lw=0.5)
for x, value in enumerate(critical_by_level.values):
    axes[0].text(x, value, f" {value}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylabel("critical buildings")
axes[0].set_title(f"{int(critical_by_level[1:].sum())} of {len(critical)} critical facilities "
                  f"reach a hazard level")

left_edge = np.zeros(len(road_km))
for value, colour in LEVELS:
    widths = road_km[LEVEL_LABEL[value]].to_numpy()
    axes[1].barh(road_km.index, widths, left=left_edge, color=colour, edgecolor="black", lw=0.4,
                 label=LEVEL_LABEL[value])
    left_edge += widths
axes[1].set_xlabel("road at a hazard level (km)")
axes[1].set_title("Road network at risk, by class")
axes[1].legend(fontsize=8)

fig.tight_layout(); plt.show()

total_km = roads.road_length_m.sum() / 1000
print(f"{left_edge.sum():,.0f} km of the island's {total_km:,.0f} km of road reaches a "
      f"hazard level ({100 * left_edge.sum() / total_km:.0f}%)")

The left panel is 528 rows of data and answers a question nobody could ask of the summary
table, because the table counts every building the same. The right panel separates the
trunk network from residential lanes, which the table adds together.

Other things this same dataframe will answer without new data — `risk_level` holds the
IBF workflow's own VERY LOW / LOW / MEDIUM / HIGH matrix, so you can compare your
classification against theirs; `subtype` splits buildings into residential, commercial,
education and medical; and the four probability columns can be crossed against each other
rather than collapsed into one level.

## 9. From notebook to dashboard

Now put it on a screen. **[Exercise 3](../../docs/Barbados/exercise_3.rst)** builds the
`Barbados Hands On 3` dashboard, and every tile in it is a section of this notebook
wrapped in a plugin:

| notebook section | dashboard tile | plugin |
|---|---|---|
| 1 — the four rasters, as URLs | four raster layers (built in Exercise 1) | none; the map reads the URLs |
| 2 and 7 — the classified grid, vectorised | the hazard map layer | `uffis_hazard_layer_barbados` |
| 4 and 7 — the classified receptors | the impact map layer | `uffis_impact_layer_barbados` |
| 6 — the impact table | the summary table | `uffis_impact_summary_barbados` |

A **plugin** is an installable Python class TethysDash discovers on its own, through an
entry point in `pyproject.toml`:

```toml
[project.entry-points."intake.drivers"]
uffis_impact_summary_barbados = "tgf_wmo_plugins.impact_summary:ImpactSummaryBarbados"
uffis_impact_layer_barbados   = "tgf_wmo_plugins.impact_layer:ImpactLayerBarbados"
uffis_hazard_layer_barbados   = "tgf_wmo_plugins.hazard_layer:HazardLayerBarbados"
```

Each family has one base class holding the computation and one subclass per country, which
sets only what differs:

```python
class ImpactSummaryBarbados(BarbadosParish, BaseImpactSummary):
    LANG = "en"
    country = "barbados"                       # selects the data tables below
    type = "table"                             # -> run() returns {"title", "data": [rows]}
    args = {**threshold_args("en"),            # low/medium/high/severe_threshold, "number"
            "parish": BARBADOS_PARISH_OPTIONS} # a list of {"value","label"} -> a dropdown
    name = "uffis_impact_summary_barbados"     # must match the entry-point key
```

`country` selects exactly what this notebook hard-coded:

```python
PROB_URLS["barbados"]        # section 1: the four rasters, shallowest first
GPKG_LAYERS["barbados"]      # section 4: buildings_ibf / roads_ibf and the p_ge_* renames
PROB_FIELDS["barbados"]      # section 4: probability_low ... probability_severe
UNIT_POPULATION["barbados"]  # section 6: the per-parish denominators
```

**The five arguments are the five things this notebook hard-coded.** `GATES` in section 2
becomes the four `*_threshold` numbers; `PARISH_PCODE` in section 7 becomes `parish`. In
the dashboard each is bound to a **variable input**, so typing a new threshold re-runs the
plugin. That is the whole mechanism: a notebook constant becomes a plugin argument becomes
a control on the screen.

Argument *names* are a published contract — they are the keys a dashboard stores. Two
silent failures worth knowing about: an argument a plugin does not declare is simply
absent at run time (it does not raise; the default is used and the control appears to do
nothing), and renaming one breaks every dashboard already bound to the old name. Internally
the gates are keyed by class value, never by name, so a translation can never reach the
classification:

```python
DEFAULT_GATES = {1: 0.3, 2: 0.2, 3: 0.1, 4: 0.15}   # gates[1], not gates["Low"]
```

**Map layers answer two questions, so they implement two methods.** `run()` fires once,
when the author adds the layer, and returns the *scaffold* — style rules and legend.
`fetch_features()` fires on every change to a bound threshold and returns the GeoJSON:

```python
class HazardLayerBarbados(BarbadosParish, BaseHazardLayer):
    type = "map_layer"
    dynamic_map_layer = True                 # this flag is what enables fetch_features()

    def run(self):                           # CONFIGURE time: style and legend, once
        builder = LayerConfigurationBuilder("Hazard classification", "GeoJSON")
        builder.set_plugin_source(self.name, self.received_args)
        ...
        return builder.build()

    def fetch_features(self):                # RUN time: sections 2 and 7, as GeoJSON
        self.send_update("Reading probability layers...", percentage_complete=10)
        layers, transform, crs = self._read(barbados_prob_urls(self.unit()))
        hazard = classify_hazard(layers, self.gates())       # section 2, unchanged
        # A map layer can only point at a URL and nothing here serves a computed raster,
        # so the classified grid is vectorised: each connected run of equal class becomes
        # one polygon. Normal and NoData are dropped -- they are most of the grid.
        return {"type": "FeatureCollection", "features": self._vectorize(...),
                "crs": {"type": "name", "properties": {"name": str(crs)}}}
```

One difference from this notebook worth noting: the plugins read each parish's **own**
raster window rather than the island mosaic, because a tile showing one parish should show
the rainfall that window was matched on. The mosaic is the right input for the island-wide
picture this notebook draws; the windows are right for one parish at a time.

**If you want your section 8 charts on a dashboard**, the route is the same. A `plotly`
plugin returns a figure dict instead of a table, and the rest of the class is what you see
above. Start from `notebooks/01_plugin_example.ipynb`, which builds the simplest possible
plugin from nothing.

### Things to try

- Set all four gates to 0.50 and re-run sections 4 to 7. You have reproduced the file's own
  `hazard_flag`. Does the map become easier or harder to explain than the 0.8 default?
- Move the gates apart instead of keeping them equal — say 0.8 / 0.6 / 0.4 / 0.3, so a
  deeper threshold needs less probability to count. Which version would you defend to a
  duty officer?
- Change `PARISH_PCODE` in section 7 to `BB01` or `BB10` and re-run. The island-wide tables
  do not change; only the view does. Why is that the right behaviour?
- Compare your `level` against the `risk_level` the workflow wrote. They answer different
  questions — a probability gate, against likelihood crossed with severity. Which belongs
  on a warning, and which on a briefing?
- Build a fourth chart in section 8. `subtype`, `building_area_m2` and the four raw
  probability columns are all sitting in the dataframe untouched.
- Run [Exercise 2](02_storm_impact.ipynb) beside this one. That one answers *"how deep in
  storm 150"*, this one *"what are the odds of 30 cm"*, over the same buildings on the same
  grid. Which would you put in front of a duty officer, and what would you say about the
  other?